# Okada Manila UAI Survey Cleaning Procedure

**Purpose:** This notebook documents the full cleaning procedure used to generate `Okada_Manila_UAI_Cleaned_Qualified_Dataset.xlsx` from `[Capstone] Okada Manila UAI Survey (Responses).xlsx`.

**Environment:** Conda `base` environment.  
**Expected interpreter:** `/opt/anaconda3/bin/python`  
**Expected qualified sample:** `n = 229`

The output workbook contains a de-identified, analysis-ready dataset, AIDA scores, AIDA component scores, multi-select indicator columns, cleaning summary, data dictionary, and frequency tables.


## 1. Load Libraries and Confirm Conda Base Environment


In [1]:
from pathlib import Path
import re
import sys

import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo

SOURCE = Path('[Capstone] Okada Manila UAI Survey (Responses).xlsx')
OUTPUT = Path('Okada_Manila_UAI_Cleaned_Qualified_Dataset.xlsx')
SHEET = 'Form Responses 1'

print('Python executable:', sys.executable)
print('Pandas version:', pd.__version__)
print('Source exists:', SOURCE.exists())


Python executable: /opt/anaconda3/bin/python
Pandas version: 3.0.3
Source exists: True


## 2. Define Cleaning and Scoring Helper Functions


In [2]:
def pct(part, total, digits=1):
    if total == 0 or pd.isna(part):
        return 0.0
    return round(float(part) / float(total) * 100, digits)


def normalize_1_5(series):
    """Convert a 1-5 scale to a 0-100 score."""
    return (pd.to_numeric(series, errors='coerce') - 1) / 4 * 100


def yes_no(series):
    """Score Yes as 100 and No as 0."""
    return series.map({'Yes': 100, 'No': 0}).astype('float')


def agreement(series):
    """Convert agreement labels to a 0-100 score."""
    values = {
        'Strongly disagree': 1,
        'Disagree': 2,
        'Neutral': 3,
        'Agree': 4,
        'Strongly agree': 5,
    }
    return normalize_1_5(series.map(values))


def ordinal(series, mapping):
    """Map ordered categories to 1-5, then normalize to 0-100."""
    return normalize_1_5(series.map(mapping))


def has_option(series, option):
    """Detect an exact multi-select option safely, including labels with punctuation."""
    return series.fillna('').astype(str).str.contains(re.escape(option), case=False, regex=True)


def count_options(df, column, options):
    """Create 0/1 indicator columns for a multi-select item."""
    out = pd.DataFrame(index=df.index)
    for option in options:
        out[option] = has_option(df[column], option).astype(int)
    return out


def freq_table(series, question_alias, response_label='response'):
    counts = series.fillna('Missing / Not applicable').value_counts(dropna=False)
    total = counts.sum()
    return pd.DataFrame({
        'question_alias': question_alias,
        response_label: counts.index,
        'frequency': counts.values,
        'percent': [pct(v, total) for v in counts.values],
    })


def multi_table(df, column, options, question_alias):
    indicators = count_options(df, column, options)
    counts = indicators.sum().sort_values(ascending=False)
    return pd.DataFrame({
        'question_alias': question_alias,
        'option': counts.index,
        'frequency': counts.values,
        'percent_of_respondents': [pct(v, len(df)) for v in counts.values],
    })


## 3. Load Raw Survey Data and Create Compact Column Aliases


In [3]:
raw = pd.read_excel(SOURCE, sheet_name=SHEET)
cols = list(raw.columns)

alias = {
    'timestamp': cols[0], 'score': cols[1], 'consent': cols[2], 'q1_children': cols[3],
    'q2_child_count': cols[4], 'q3_child_ages': cols[5], 'q4_family_situation': cols[6],
    'q5_staycation_behavior': cols[7], 'q6_amenities': cols[8], 'q7_activities': cols[9],
    'q8_family_amenity_considered': cols[10], 'q9_activity_interest': cols[11],
    'q10_event_awareness': cols[12], 'q11_promo_channels': cols[13],
    'q12_themed_experience': cols[14], 'q13_staycation_purpose': cols[15],
    'q14_amenity_importance': cols[16], 'q15_booking_motivators': cols[17],
    'q16_emotional_appeal': cols[18], 'q17_budget_general': cols[19],
    'q18_event_participation': cols[20], 'q19_event_driver': cols[21],
    'q20_package_interest': cols[22], 'q21_package_book': cols[23],
    'q22_family_friendly_luxury_likelihood': cols[24], 'q23_ir_awareness': cols[25],
    'q24_ir_preference': cols[26], 'q25_ir_choice_reason': cols[27],
    'q26_okada_familiarity': cols[28], 'q27_okada_sources': cols[29],
    'q28_competitor': cols[30], 'q29_okada_offerings': cols[31],
    'q30_stayed_okada': cols[32], 'q31a_booking_method_past': cols[33],
    'q31b_past_visits': cols[34], 'q31c_planning_staycation': cols[35],
    'q31d_booking_method_planned': cols[36], 'q31e_planned_visits': cols[37],
    'q32_stay_length': cols[38], 'q33_room_type': cols[39], 'q34_okada_budget': cols[40],
    'q35_booking_factor': cols[41], 'q36_okada_family_package': cols[42],
    'q37_return': cols[43], 'q37_events_interest': cols[44],
    'q37_experiential_interest': cols[45], 'nickname': cols[46], 'age': cols[47],
    'gender': cols[48], 'nationality': cols[49], 'education': cols[50],
    'occupation': cols[51], 'income': cols[52], 'location': cols[53],
    'social_platform': cols[54], 'gcash': cols[55],
}

data_dictionary = pd.DataFrame([
    {'clean_alias': short, 'original_column': original.replace('\n', ' ')}
    for short, original in alias.items()
])

print('Raw shape:', raw.shape)
display(data_dictionary.head(10))


Raw shape: (301, 56)


,clean_alias,original_column
0,timestamp,Timestamp
1,score,Score
2,consent,Confidentiality and Data Privacy
3,q1_children,Q1. Do you have children?
4,q2_child_count,Q2. How many children do you have?
5,q3_child_ages,Q3. What are the ages of your children? If y...
6,q4_family_situation,Q4. Which best describes your current family s...
7,q5_staycation_behavior,Q5. Which statement best describes your family...
8,q6_amenities,"Q6. When planning to go out as a family, which..."
9,q7_activities,Q7. When visiting a hotel or resort with your ...


## 4. Apply Qualification Rule and De-Identify Record-Level Data


In [4]:
# Main qualification rule: retain respondents who answered Q1 = Yes.
qualified_original = raw[raw[alias['q1_children']].eq('Yes')].copy().reset_index(drop=True)

# Rename columns to compact aliases.
reverse_alias = {original: short for short, original in alias.items()}
qualified = qualified_original.rename(columns=reverse_alias)
qualified.insert(0, 'respondent_id', [f'R{idx:03d}' for idx in range(1, len(qualified) + 1)])

# De-identification: remove direct identifiers and non-analysis fields.
removed_columns = ['timestamp', 'score', 'nickname', 'gcash']
clean = qualified.drop(columns=[c for c in removed_columns if c in qualified.columns])

cleaning_summary = pd.DataFrame([
    ['source_workbook', SOURCE.name],
    ['source_sheet', SHEET],
    ['raw_response_rows', len(raw)],
    ['raw_columns', len(raw.columns)],
    ['qualified_filter', 'q1_children == Yes'],
    ['qualified_rows', len(clean)],
    ['excluded_rows_without_q1_yes', len(raw) - len(clean)],
    ['qualified_rows_with_no_consent_response', int((qualified['consent'] == 'No, I do not give my consent.').sum())],
    ['exact_duplicate_rows_in_qualified_sample', int(qualified_original.duplicated().sum())],
    ['removed_from_record_level_cleaned_data', ', '.join(removed_columns)],
], columns=['item', 'value'])

display(cleaning_summary)
print('Clean record-level shape before AIDA scores:', clean.shape)


,item,value
0,source_workbook,[Capstone] Okada Manila UAI Survey (Responses)...
1,source_sheet,Form Responses 1
2,raw_response_rows,301
3,raw_columns,56
4,qualified_filter,q1_children == Yes
5,qualified_rows,229
6,excluded_rows_without_q1_yes,72
7,qualified_rows_with_no_consent_response,2
8,exact_duplicate_rows_in_qualified_sample,0
9,removed_from_record_level_cleaned_data,"timestamp, score, nickname, gcash"


Clean record-level shape before AIDA scores: (229, 53)


## 5. Define Ordered Mappings and Multi-Select Option Lists


In [5]:
price_map = {
    'Below ₱10,000': 1,
    '₱10,000–₱14,999': 2,
    '₱15,000–₱19,999': 3,
    '₱20,000–₱29,999': 4,
    '₱30,000 and above': 5,
}

behavior_map = {
    'We are not interested in hotel staycations.': 1,
    'We rarely stay in hotels/resorts.': 2,
    'We are planning to stay in a hotel/resort within the next 12 months.': 4,
    'We have stayed in a hotel/resort within the past 12 months.': 5,
}

frequency_map = {
    "I haven't visited in the past 12 months": 1,
    'Once': 2,
    '2–3 times': 3,
    '4–5 times': 4,
    'More than 5 times': 5,
}

q6_options = ['Swimming pool', "Kids' play area", 'Restaurants', 'Family rooms', 'Events / themed activities', 'Shopping areas', 'Entertainment shows', 'Spa / wellness', 'Outdoor activities']
q7_options = ['Swimming', 'Dining', 'Staycation', 'Relaxation', 'Bonding with family', 'Taking photos / social media content', 'Watching entertainment shows', 'Shopping', 'Attending events', 'Fountain show']
q11_options = ['Social media (Facebook, Instagram, TikTok, YouTube)', 'Influencers / content creators', 'Friends or relatives', 'Online advertisements', 'Hotel websites', 'News or online articles', 'Online Travel Agencies (OTA) (e.g. Klook, Agoda, Trip.com)']
q15_options = ['Promotions / discounts', 'Family packages', 'Amenities', 'Brand reputation', 'Recommendations', 'Convenience / location', 'Entertainment offerings']
q20_options = ['Staycation + Buffet dining', 'Staycation + Kids’ activity passes', 'Staycation + Spa or wellness access', 'Staycation + Entertainment show tickets', 'Staycation + Shopping vouchers', 'Staycation + Event access', 'Staycation + Family photo package']
q25_options = ['Family-friendly atmosphere', 'Amenities for children', 'Safety and comfort', 'Room quality', 'Dining options', 'Entertainment offerings', 'Promotions/packages', 'Brand reputation', 'Location', 'Luxury experience']
q27_options = ['Facebook', 'Instagram', 'TikTok', 'YouTube', 'Influencers / vloggers', 'Friends or family', 'Traditional advertisements (TV, newspaper, magazine, radio)', 'Out-of-Home advertisements (Billboard, LED Billboard, Cinema)', 'Online articles', 'Online Travel Agencies (OTA) (e.g. Klook, Agoda, Trip.com)', 'Events / Travel expo']
q29_options = ['Hotel staycations', 'Restaurants and dining', 'Swimming pools', 'Attractions (e.g. The Fountain)', 'Shopping areas', 'Entertainment shows', 'Seasonal events', 'Family activities (e.g. PLAY Kids Club, Thrillscape)', 'Wellness / spa (e.g. The Retreat Spa)', 'Events / MICE (Meetings, Incentives, Conferences, Exhibits) facilities']

print('Option lists prepared for multi-select parsing.')


Option lists prepared for multi-select parsing.


## 6. Compute AIDA Component Scores and Overall AIDA Index


In [6]:
attention_parts = pd.DataFrame({'respondent_id': clean['respondent_id']})
attention_parts['attention_ir_awareness_okada'] = has_option(qualified_original[alias['q23_ir_awareness']], 'Okada Manila').astype(float) * 100
attention_parts['attention_okada_familiarity'] = normalize_1_5(qualified_original[alias['q26_okada_familiarity']])
attention_parts['attention_okada_media_exposure_breadth'] = count_options(qualified_original, alias['q27_okada_sources'], q27_options).sum(axis=1) / len(q27_options) * 100
attention_parts['attention_okada_offering_awareness_breadth'] = count_options(qualified_original, alias['q29_okada_offerings'], q29_options).sum(axis=1) / len(q29_options) * 100

interest_parts = pd.DataFrame({'respondent_id': clean['respondent_id']})
interest_parts['interest_family_amenity_consideration'] = yes_no(qualified_original[alias['q8_family_amenity_considered']])
interest_parts['interest_family_amenity_importance'] = normalize_1_5(qualified_original[alias['q14_amenity_importance']])
interest_parts['interest_prior_family_event_participation'] = yes_no(qualified_original[alias['q18_event_participation']])
interest_parts['interest_package_interest_breadth'] = count_options(qualified_original, alias['q20_package_interest'], q20_options).sum(axis=1) / len(q20_options) * 100

desire_parts = pd.DataFrame({'respondent_id': clean['respondent_id']})
desire_parts['desire_likelihood_family_friendly_luxury'] = normalize_1_5(qualified_original[alias['q22_family_friendly_luxury_likelihood']])
desire_parts['desire_okada_stay_budget'] = ordinal(qualified_original[alias['q34_okada_budget']], price_map)
desire_parts['desire_family_oriented_events_interest'] = agreement(qualified_original[alias['q37_events_interest']])
desire_parts['desire_experiential_family_package_interest'] = agreement(qualified_original[alias['q37_experiential_interest']])

action_parts = pd.DataFrame({'respondent_id': clean['respondent_id']})
action_parts['action_hotel_staycation_readiness'] = ordinal(qualified_original[alias['q5_staycation_behavior']], behavior_map)
action_parts['action_prior_okada_hotel_stay'] = yes_no(qualified_original[alias['q30_stayed_okada']])
action_parts['action_planned_okada_family_staycation'] = yes_no(qualified_original[alias['q31c_planning_staycation']])
action_parts['action_past_okada_visit_frequency'] = ordinal(qualified_original[alias['q31b_past_visits']], frequency_map)
action_parts['action_planned_okada_visit_frequency'] = ordinal(qualified_original[alias['q31e_planned_visits']], frequency_map)
action_parts['action_return_intention_family_occasions'] = agreement(qualified_original[alias['q37_return']])

scores = pd.DataFrame({'respondent_id': clean['respondent_id']})
scores['attention_score'] = attention_parts.drop(columns='respondent_id').mean(axis=1, skipna=True)
scores['interest_score'] = interest_parts.drop(columns='respondent_id').mean(axis=1, skipna=True)
scores['desire_score'] = desire_parts.drop(columns='respondent_id').mean(axis=1, skipna=True)
scores['action_score'] = action_parts.drop(columns='respondent_id').mean(axis=1, skipna=True)
scores['aida_overall'] = scores[['attention_score', 'interest_score', 'desire_score', 'action_score']].mean(axis=1)

score_cols = ['attention_score', 'interest_score', 'desire_score', 'action_score', 'aida_overall']
clean_with_scores = clean.merge(scores, on='respondent_id', how='left')
components = (
    attention_parts
    .merge(interest_parts, on='respondent_id')
    .merge(desire_parts, on='respondent_id')
    .merge(action_parts, on='respondent_id')
    .merge(scores, on='respondent_id')
)

stage_summary = scores[score_cols].describe().T[['count','mean','std','min','25%','50%','75%','max']].round(2).reset_index().rename(columns={'index':'score'})
component_summary = pd.concat({
    'Attention': attention_parts.drop(columns='respondent_id').mean().round(2),
    'Interest': interest_parts.drop(columns='respondent_id').mean().round(2),
    'Desire': desire_parts.drop(columns='respondent_id').mean().round(2),
    'Action': action_parts.drop(columns='respondent_id').mean().round(2),
}).reset_index().rename(columns={'level_0':'aida_stage','level_1':'component',0:'mean_score'})

display(stage_summary)


,score,count,mean,std,min,25%,50%,75%,max
0,attention_score,229.0,65.94,9.83,32.27,60.57,66.82,71.36,100.00
1,interest_score,229.0,68.44,15.32,25.89,54.46,75.00,82.14,100.00
2,desire_score,229.0,69.60,13.61,31.25,62.50,68.75,81.25,100.00
3,action_score,229.0,62.25,16.21,25.00,50.00,62.50,75.00,100.00
4,aida_overall,229.0,66.56,8.42,44.90,60.42,66.00,71.51,96.88


## 7. Create Frequency Tables and Multi-Select Indicator Sheets


In [7]:
frequency_tables = pd.concat([
    freq_table(qualified_original[alias['age']], 'age'),
    freq_table(qualified_original[alias['gender']], 'gender'),
    freq_table(qualified_original[alias['income']], 'income'),
    freq_table(qualified_original[alias['social_platform']], 'social_platform'),
    freq_table(qualified_original[alias['q24_ir_preference']], 'q24_ir_preference'),
    freq_table(qualified_original[alias['q28_competitor']], 'q28_competitor'),
    freq_table(qualified_original[alias['q30_stayed_okada']], 'q30_stayed_okada'),
    freq_table(qualified_original[alias['q31c_planning_staycation']], 'q31c_planning_staycation'),
    freq_table(qualified_original[alias['q33_room_type']], 'q33_room_type'),
    freq_table(qualified_original[alias['q34_okada_budget']], 'q34_okada_budget'),
    freq_table(qualified_original[alias['q35_booking_factor']], 'q35_booking_factor'),
    freq_table(qualified_original[alias['q36_okada_family_package']], 'q36_okada_family_package'),
], ignore_index=True)

multi_select_counts = pd.concat([
    multi_table(qualified_original, alias['q6_amenities'], q6_options, 'q6_amenities'),
    multi_table(qualified_original, alias['q7_activities'], q7_options, 'q7_activities'),
    multi_table(qualified_original, alias['q11_promo_channels'], q11_options, 'q11_promo_channels'),
    multi_table(qualified_original, alias['q15_booking_motivators'], q15_options, 'q15_booking_motivators'),
    multi_table(qualified_original, alias['q20_package_interest'], q20_options, 'q20_package_interest'),
    multi_table(qualified_original, alias['q25_ir_choice_reason'], q25_options, 'q25_ir_choice_reason'),
    multi_table(qualified_original, alias['q27_okada_sources'], q27_options, 'q27_okada_sources'),
    multi_table(qualified_original, alias['q29_okada_offerings'], q29_options, 'q29_okada_offerings'),
], ignore_index=True)

indicator_frames = []
for prefix, col, opts in [
    ('q6', alias['q6_amenities'], q6_options),
    ('q7', alias['q7_activities'], q7_options),
    ('q11', alias['q11_promo_channels'], q11_options),
    ('q20', alias['q20_package_interest'], q20_options),
    ('q27', alias['q27_okada_sources'], q27_options),
    ('q29', alias['q29_okada_offerings'], q29_options),
]:
    ind = count_options(qualified_original, col, opts)
    ind.columns = [
        prefix + '_' + re.sub(r'[^a-z0-9]+', '_', option.lower()).strip('_')[:55]
        for option in ind.columns
    ]
    indicator_frames.append(ind)

indicators = pd.concat(indicator_frames, axis=1)
indicators.insert(0, 'respondent_id', clean['respondent_id'])

display(frequency_tables.head(10))
display(multi_select_counts.head(10))


,question_alias,response,frequency,percent
0,age,25 to 34 years old,83,36.2
1,age,35 to 44 years old,80,34.9
2,age,45 to 54 years old,35,15.3
3,age,18 to 24 years old,17,7.4
4,age,54 to 64 years old,11,4.8
5,age,65 and above,3,1.3
6,gender,Woman,122,53.3
7,gender,Man,106,46.3
8,gender,Prefer not to say,1,0.4
9,income,"₱77,000 – ₱131,999",63,27.5


,question_alias,option,frequency,percent_of_respondents
0,q6_amenities,Swimming pool,173,75.5
1,q6_amenities,Restaurants,168,73.4
2,q6_amenities,Kids' play area,157,68.6
3,q6_amenities,Family rooms,108,47.2
4,q6_amenities,Shopping areas,69,30.1
5,q6_amenities,Events / themed activities,62,27.1
6,q6_amenities,Spa / wellness,61,26.6
7,q6_amenities,Entertainment shows,60,26.2
8,q6_amenities,Outdoor activities,55,24.0
9,q7_activities,Swimming,169,73.8


## 8. Build Data Dictionary and Export Cleaned Excel Workbook


In [8]:
computed_dictionary = pd.DataFrame([
    {'clean_alias':'respondent_id','original_column':'Generated anonymous sequential respondent identifier','included_in_cleaned_dataset':'Yes'},
    {'clean_alias':'attention_score','original_column':'Computed AIDA Attention score, 0-100','included_in_cleaned_dataset':'Yes'},
    {'clean_alias':'interest_score','original_column':'Computed AIDA Interest score, 0-100','included_in_cleaned_dataset':'Yes'},
    {'clean_alias':'desire_score','original_column':'Computed AIDA Desire score, 0-100','included_in_cleaned_dataset':'Yes'},
    {'clean_alias':'action_score','original_column':'Computed AIDA Action score, 0-100','included_in_cleaned_dataset':'Yes'},
    {'clean_alias':'aida_overall','original_column':'Computed mean of four AIDA stage scores, 0-100','included_in_cleaned_dataset':'Yes'},
])

source_dictionary = pd.DataFrame([{
    'clean_alias': short,
    'original_column': original.replace('\n', ' '),
    'included_in_cleaned_dataset': 'No' if short in removed_columns else ('Yes' if short in clean_with_scores.columns else 'Supporting / computed'),
} for short, original in alias.items()])

data_dictionary = pd.concat([computed_dictionary, source_dictionary], ignore_index=True)

with pd.ExcelWriter(OUTPUT, engine='openpyxl') as writer:
    clean_with_scores.to_excel(writer, sheet_name='Cleaned_Qualified_Data', index=False)
    components.to_excel(writer, sheet_name='AIDA_Components', index=False)
    indicators.to_excel(writer, sheet_name='MultiSelect_Indicators', index=False)
    cleaning_summary.to_excel(writer, sheet_name='Cleaning_Summary', index=False)
    data_dictionary.to_excel(writer, sheet_name='Data_Dictionary', index=False)
    stage_summary.to_excel(writer, sheet_name='AIDA_Stage_Summary', index=False)
    component_summary.to_excel(writer, sheet_name='AIDA_Component_Summary', index=False)
    frequency_tables.to_excel(writer, sheet_name='Frequency_Tables', index=False)
    multi_select_counts.to_excel(writer, sheet_name='MultiSelect_Counts', index=False)

print('Workbook exported:', OUTPUT.resolve())


Workbook exported: /Users/freshliannes.rosal/Downloads/Rhuss Project/Okada_Manila_UAI_Cleaned_Qualified_Dataset.xlsx


## 9. Format Workbook Sheets for Readability


In [9]:
wb = load_workbook(OUTPUT)
header_fill = PatternFill('solid', fgColor='17324D')
header_font = Font(color='FFFFFF', bold=True)
thin = Side(style='thin', color='D6DEE8')
border = Border(left=thin, right=thin, top=thin, bottom=thin)

for ws in wb.worksheets:
    ws.freeze_panes = 'A2'
    ws.sheet_view.showGridLines = False
    max_row, max_col = ws.max_row, ws.max_column
    if max_row >= 1 and max_col >= 1:
        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            cell.border = border
        for row in ws.iter_rows(min_row=2, max_row=max_row, max_col=max_col):
            for cell in row:
                cell.border = border
                cell.alignment = Alignment(vertical='top', wrap_text=True)
        ref = f'A1:{get_column_letter(max_col)}{max_row}'
        safe_name = re.sub(r'[^A-Za-z0-9_]', '_', ws.title)[:20]
        table = Table(displayName=f'Tbl_{safe_name}', ref=ref)
        style = TableStyleInfo(
            name='TableStyleMedium2',
            showFirstColumn=False,
            showLastColumn=False,
            showRowStripes=True,
            showColumnStripes=False,
        )
        table.tableStyleInfo = style
        ws.add_table(table)
    for col_idx in range(1, max_col + 1):
        letter = get_column_letter(col_idx)
        values = [str(ws.cell(row=r, column=col_idx).value or '') for r in range(1, min(max_row, 80) + 1)]
        width = min(max(max(len(v) for v in values) + 2, 10), 44)
        if ws.title == 'Cleaned_Qualified_Data' and col_idx > 8:
            width = min(width, 30)
        ws.column_dimensions[letter].width = width
    ws.row_dimensions[1].height = 36

# Place the audit sheet first.
wb._sheets = [wb['Cleaning_Summary']] + [sheet for sheet in wb._sheets if sheet.title != 'Cleaning_Summary']
wb.save(OUTPUT)

print('Workbook formatting applied.')


Workbook formatting applied.


## 10. Validation Checks


In [10]:
check = pd.read_excel(OUTPUT, sheet_name='Cleaned_Qualified_Data')
wb_check = load_workbook(OUTPUT, read_only=True)

checks = {
    'source_shape_is_301_by_56': raw.shape == (301, 56),
    'qualified_rows_equal_229': len(check) == 229,
    'excluded_rows_equal_72': len(raw) - len(check) == 72,
    'direct_identifiers_removed': all(col not in check.columns for col in ['nickname', 'gcash', 'timestamp', 'score']),
    'aida_scores_bounded_0_to_100': check[score_cols].apply(lambda s: s.between(0, 100) | s.isna()).all().all(),
    'expected_sheets_present': set([
        'Cleaning_Summary', 'Cleaned_Qualified_Data', 'AIDA_Components', 'MultiSelect_Indicators',
        'Data_Dictionary', 'AIDA_Stage_Summary', 'AIDA_Component_Summary',
        'Frequency_Tables', 'MultiSelect_Counts'
    ]).issubset(set(wb_check.sheetnames)),
}

validation = pd.DataFrame({
    'check': list(checks.keys()),
    'status': ['PASS' if value else 'REVIEW' for value in checks.values()],
})
display(validation)

if not all(checks.values()):
    raise AssertionError('One or more validation checks need review.')

print('Final cleaned workbook:', OUTPUT.resolve())
print('Rows:', len(check), '| Columns:', len(check.columns))
print('Sheets:', ', '.join(wb_check.sheetnames))


,check,status
0,source_shape_is_301_by_56,PASS
1,qualified_rows_equal_229,PASS
2,excluded_rows_equal_72,PASS
3,direct_identifiers_removed,PASS
4,aida_scores_bounded_0_to_100,PASS
5,expected_sheets_present,PASS


Final cleaned workbook: /Users/freshliannes.rosal/Downloads/Rhuss Project/Okada_Manila_UAI_Cleaned_Qualified_Dataset.xlsx
Rows: 229 | Columns: 58
Sheets: Cleaning_Summary, Cleaned_Qualified_Data, AIDA_Components, MultiSelect_Indicators, Data_Dictionary, AIDA_Stage_Summary, AIDA_Component_Summary, Frequency_Tables, MultiSelect_Counts
